# 5 Model Quantization for Edge Deployment
This notebook demonstrates Post-Training Quantization (PTQ) to reduce model size and latency using PyTorch.
The model is loaded from BentoML and all metrics are tracked via MLflow.

In [ ]:
import os
import time
import tempfile

import torch
import bentoml
import mlflow

from pneumonia_classifier.ml.model.arch import Net

torch.serialization.add_safe_globals([Net])

## Load Model from BentoML

In [2]:
print("Loading model from BentoML...")
bento_model = bentoml.pytorch.get("pneumonia_classifier_aug_augmented_heavy:wypxe3qwqsx75ahb")
model = bentoml.pytorch.load_model(bento_model, weights_only=False)
model.eval()
print(f"Loaded: {bento_model.tag}")

Loading model from BentoML...


C:\Users\user\AppData\Local\Temp\ipykernel_14316\804820423.py:2: BentoMLDeprecationWarning: `bentoml.pytorch` is deprecated since v1.4 and will be removed in a future version.
  bento_model = bentoml.pytorch.get("pneumonia_classifier_aug_augmented_heavy:wypxe3qwqsx75ahb")


Loaded: pneumonia_classifier_aug_augmented_heavy:wypxe3qwqsx75ahb


In [ ]:
import torch
print(torch.backends.quantized.supported_engines)

## Dynamic Quantization (INT8)
Quantizes the weights of linear layers (and optionally others) to INT8.

In [16]:
import torch
import torch.ao.quantization as quant
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

torch.backends.quantized.engine = "onednn"

model.eval()

# Required example input
example_input = torch.randn(1, 3, 224, 224)

# QConfig
qconfig = quant.get_default_qconfig("onednn")
qconfig_dict = {"": qconfig}

# Prepare (graph rewrite)
prepared = prepare_fx(model, qconfig_dict, example_input)

# Calibration
with torch.no_grad():
    for _ in range(20):
        prepared(torch.randn(1, 3, 224, 224))

# Convert
quantized_model = convert_fx(prepared)
quantized_model.eval()

C:\Users\user\AppData\Local\Temp\ipykernel_14316\2279640467.py:13: UserWarning: Default qconfig of oneDNN backend with reduce_range of false may have accuracy issues on CPU without Vector Neural Network Instruction support.
  qconfig = quant.get_default_qconfig("onednn")
C:\Users\user\AppData\Local\Temp\ipykernel_14316\2279640467.py:17: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for mor

GraphModule(
  (convolution_block1): Module(
    (0): QuantizedConvReLU2d(3, 8, kernel_size=(3, 3), stride=(1, 1), scale=0.009978815913200378, zero_point=0)
    (2): QuantizedBatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolution_block2): Module(
    (0): QuantizedConvReLU2d(8, 20, kernel_size=(3, 3), stride=(1, 1), scale=0.010164931416511536, zero_point=0)
    (2): QuantizedBatchNorm2d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling22): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolution_block3): Module(
    (0): QuantizedConvReLU2d(20, 10, kernel_size=(1, 1), stride=(1, 1), scale=0.009821256622672081, zero_point=0)
    (2): QuantizedBatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling33): MaxPool2d(kernel_size=2, stride=2, padding=0, dilati

## Size Comparison

In [17]:
def get_model_size_mb(m):
    """Save model state_dict to a temp file and return its size in MB."""
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
        torch.save(m.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024)
    os.remove(f.name)
    return size_mb

fp32_size = get_model_size_mb(model)
int8_size = get_model_size_mb(quantized_model)
compression_ratio = fp32_size / int8_size if int8_size > 0 else 0

print(f"FP32 Model Size: {fp32_size:.4f} KB")
print(f"INT8 Model Size: {int8_size:.4f} KB")
print(f"Compression Ratio: {compression_ratio:.2f}x")

FP32 Model Size: 243.4854 KB
INT8 Model Size: 52.4541 KB
Compression Ratio: 4.64x


In [18]:
dummy = torch.randn(1, 3, 224, 224)

fp32_out = model(dummy)  # original model
int8_out = quantized_model(dummy)

print("Mean abs diff:", (fp32_out - int8_out).abs().mean().item())

Mean abs diff: 0.27367037534713745


## Latency Benchmark

In [19]:
dummy_input = torch.randn(1, 3, 224, 224)
NUM_RUNS = 100

def benchmark(m, input_data, num_runs=100):
    """Returns average latency in ms."""
    # Warmup
    for _ in range(10):
        m(input_data)
    start = time.time()
    for _ in range(num_runs):
        m(input_data)
    end = time.time()
    return (end - start) / num_runs * 1000

fp32_latency = benchmark(model, dummy_input, NUM_RUNS)
int8_latency = benchmark(quantized_model, dummy_input, NUM_RUNS)
speedup = fp32_latency / int8_latency if int8_latency > 0 else 0

print(f"FP32 average latency: {fp32_latency:.2f} ms")
print(f"INT8 Dynamic average latency: {int8_latency:.2f} ms")
print(f"Speedup: {speedup:.2f}x")

FP32 average latency: 108.12 ms
INT8 Dynamic average latency: 27.11 ms
Speedup: 3.99x


## Log Results to MLflow & Save Quantized Model to BentoML

In [20]:
mlflow.set_experiment("model_quantization")

with mlflow.start_run(run_name="ptq_int8_dynamic"):
    mlflow.log_params({
        "quantization_type": "dynamic",
        "target_dtype": "qint8",
        "source_model": str(bento_model.tag),
        "benchmark_runs": NUM_RUNS
    })

    mlflow.log_metrics({
        "fp32_size_mb": fp32_size,
        "int8_size_mb": int8_size,
        "compression_ratio": compression_ratio,
        "fp32_latency_ms": fp32_latency,
        "int8_latency_ms": int8_latency,
        "speedup": speedup
    })

    # Save the quantized model to BentoML
    quantized_bento = bentoml.pytorch.save_model(
        "pneumonia_classifier_cnn_int8",
        quantized_model,
        signatures={
            "__call__": {"batchable": True, "batch_dim": 0}
        }
    )
    mlflow.log_param("quantized_bento_tag", str(quantized_bento.tag))
    print(f"Quantized model saved to BentoML: {quantized_bento.tag}")
    print("All metrics logged to MLflow.")

Quantized model saved to BentoML: pneumonia_classifier_cnn_int8:ra32tviwyo4u3ahb
All metrics logged to MLflow.
